# 06 — Consolidated Modeling (team final)

The agreed team modeling notebook, merging the best of both modeling drafts:

- **Sem's setup** — fixed-date chronological split, 17 predictors (one-hot day-of-week), a tuned Random Forest, and **WAPE** plus a **long-stockout sensitivity check**.
- **The group's additions** — **permutation importance** and a **Phase 5 evaluation on the dataset's official eval split** (the true next 7 days).

**Target:** next-day `sale_amount` (observed), predicted only from information known the day before. Same 100-series / 9,000-row subset (`data/subset/freshretail_subset_daily.csv`).

> This supersedes the split differences between the two earlier drafts, so the team reports one consistent set of numbers.

In [ ]:
# --- run from anywhere: switch to the data-prep folder so relative paths resolve ---
# (notebooks live here, but the shared data/subset and Data/ folders are under
#  'Data Prep - Derrick'; this makes data/subset/..., ../Data/..., outputs/... all work.)
import os
from pathlib import Path
_cwd = Path.cwd().resolve()
_root = next((p for p in (_cwd, *_cwd.parents) if (p / 'Data Prep - Derrick').is_dir()), None)
if _root is not None:
    os.chdir(_root / 'Data Prep - Derrick')
print('working dir:', os.getcwd())

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error

os.makedirs('outputs/models', exist_ok=True)

def rmse(y, p):
    return np.sqrt(mean_squared_error(y, p))

def wape(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return np.abs(y - p).sum() / y.sum() * 100

df = pd.read_csv('data/subset/freshretail_subset_daily.csv', parse_dates=['dt'])
df = df.sort_values(['store_id', 'product_id', 'dt']).reset_index(drop=True)
print('rows', len(df), '| series', df.groupby(['store_id', 'product_id']).ngroups)

FileNotFoundError: [Errno 2] No such file or directory: 'data/subset/freshretail_subset_daily.csv'

## 1. Predictors (no leakage)

Every predictor uses only information known **the day before** the predicted day: planned `discount` / `activity_flag` / `holiday_flag`, one-hot **day of week**, past sales (`lag1`, `lag7`, 7-day average), and **yesterday's** weather and stockout hours. Same-day weather and stockouts are excluded.

In [ ]:
series = df.groupby(['store_id', 'product_id'], observed=True)
df['sales_lag1'] = series['sale_amount'].shift(1)
df['sales_lag7'] = series['sale_amount'].shift(7)
df['sales_avg7'] = series['sale_amount'].transform(lambda s: s.shift(1).rolling(7).mean())
for col in ['stock_hour6_22_cnt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'precpt']:
    df['prev_' + col] = series[col].shift(1)
df['day_of_week'] = df['dt'].dt.day_of_week
df = pd.get_dummies(df, columns=['day_of_week'], prefix='dow', drop_first=True, dtype=int)
dow_cols = [c for c in df.columns if c.startswith('dow_')]

features = (['discount', 'activity_flag', 'holiday_flag'] + dow_cols
            + ['sales_lag1', 'sales_lag7', 'sales_avg7']
            + ['prev_stock_hour6_22_cnt', 'prev_avg_temperature', 'prev_avg_humidity',
               'prev_avg_wind_level', 'prev_precpt'])
print(len(features), 'predictors')

## 2. Split in time order (fixed calendar dates)

Train Apr 4 – May 28, validate May 29 – Jun 11, test Jun 12 – 25, 2024. The first 7 days only supply history for the 7-day features.

In [ ]:
model_df = df.dropna(subset=features).copy()
train = model_df[model_df['dt'] <= '2024-05-28']
valid = model_df[(model_df['dt'] >= '2024-05-29') & (model_df['dt'] <= '2024-06-11')]
test  = model_df[model_df['dt'] >= '2024-06-12']
for name, part in [('Train', train), ('Validation', valid), ('Test', test)]:
    print(f"{name:<11} {part['dt'].min().date()} to {part['dt'].max().date()}  rows: {len(part)}")

## 3. Measures

**MAE** (average error), **RMSE** (penalizes big misses), **WAPE** (total error as a percent of total sales). Lower is better.

In [ ]:
def score(actual, predicted):
    return {'MAE': mean_absolute_error(actual, predicted),
            'RMSE': rmse(actual, predicted),
            'WAPE (%)': wape(actual, predicted)}

## 4. Tune the Random Forest on the validation period

Grid over `max_depth` and `min_samples_leaf`; keep the lowest validation MAE. A high `min_samples_leaf` regularizes a deep tree so it does not overfit.

In [ ]:
grid = []
for depth in [5, 10, 15, None]:
    for leaf in [1, 5, 10, 20]:
        rf_try = RandomForestRegressor(n_estimators=300, max_depth=depth, min_samples_leaf=leaf,
                                       random_state=42, n_jobs=-1).fit(train[features], train['sale_amount'])
        grid.append({'max_depth': depth, 'min_samples_leaf': leaf,
                     **score(valid['sale_amount'], rf_try.predict(valid[features]))})
tuning = pd.DataFrame(grid).sort_values('MAE')
best = tuning.iloc[0]
best_depth = None if pd.isna(best['max_depth']) else int(best['max_depth'])
best_leaf = int(best['min_samples_leaf'])
print('best: max_depth', best_depth, ', min_samples_leaf', best_leaf)
tuning.head(5).round(3)

## 5. Final models and test results

Refit on train + validation, score once on the test period, and also report results with the **long-stockout days removed** (>= 12 out-of-stock business hours), where observed sales are limited by empty shelves rather than low demand.

In [ ]:
train_valid = pd.concat([train, valid])
lin = LinearRegression().fit(train_valid[features], train_valid['sale_amount'])
rf = RandomForestRegressor(n_estimators=300, max_depth=best_depth, min_samples_leaf=best_leaf,
                           random_state=42, n_jobs=-1).fit(train_valid[features], train_valid['sale_amount'])

preds = {'Baseline (prev-week same-day)': test['sales_lag7'].values,
         'Linear regression': lin.predict(test[features]),
         'Random Forest': rf.predict(test[features])}
keep = (test['stock_hour6_22_cnt'] < 12).values
print('long-stockout test rows (>=12h):', (~keep).sum(), 'of', len(test))

rows = []
for name, p in preds.items():
    alld = score(test['sale_amount'], p)
    clean = score(test['sale_amount'].values[keep], p[keep])
    rows.append({'Model': name, **alld,
                 'MAE (no long stockout)': clean['MAE'], 'WAPE% (no long stockout)': clean['WAPE (%)']})
test_results = pd.DataFrame(rows).set_index('Model').round(3)
print(test_results.to_string())

In [ ]:
order = list(preds.keys())
vals = [test_results.loc[m, 'MAE'] for m in order]
plt.figure(figsize=(7, 4))
bars = plt.bar(['Baseline', 'Linear', 'Random Forest'], vals, color=['#bbbbbb', '#86b6ef', '#2a78d6'], width=0.55)
for b, v in zip(bars, vals):
    plt.text(b.get_x() + b.get_width()/2, v + 0.008, f'{v:.3f}', ha='center', va='bottom')
plt.ylabel('Test MAE (lower is better)'); plt.title('Random Forest has the lowest test error')
plt.tight_layout(); plt.savefig('outputs/models/final_01_model_comparison.png', dpi=150); plt.close()
print('saved outputs/models/final_01_model_comparison.png')

## 6. Which factors matter most? (permutation importance)

Shuffle one predictor at a time and measure how much the **test MAE** worsens. More robust than the tree's built-in importance, which is biased toward high-cardinality features.

In [ ]:
perm = permutation_importance(rf, test[features], test['sale_amount'],
                              scoring='neg_mean_absolute_error', n_repeats=10, random_state=42, n_jobs=1)
nice = {'sales_avg7': 'Avg sales, last 7 days', 'holiday_flag': 'Holiday', 'sales_lag1': 'Sales yesterday',
        'sales_lag7': 'Sales same day last week', 'discount': 'Discount', 'activity_flag': 'Promotion',
        'prev_stock_hour6_22_cnt': 'Stockout hrs yesterday', 'prev_avg_temperature': 'Temp yesterday',
        'prev_avg_humidity': 'Humidity yesterday', 'prev_avg_wind_level': 'Wind yesterday', 'prev_precpt': 'Rain yesterday'}
imp = pd.Series(perm.importances_mean, index=features)
imp = imp.rename(lambda c: 'Day of week' if c.startswith('dow_') else nice[c]).groupby(level=0).sum().sort_values(ascending=False)
print(imp.round(4).to_string())

top = imp.head(8)[::-1]
colors = ['#2a78d6' if n in ('Holiday', 'Discount') else '#bbbbbb' for n in top.index]
plt.figure(figsize=(7, 4.2)); plt.barh(top.index, top.values, color=colors, height=0.6)
plt.xlabel('Increase in test MAE when shuffled'); plt.title('Recent sales dominate; holidays & discounts next')
plt.tight_layout(); plt.savefig('outputs/models/final_02_feature_importance.png', dpi=150); plt.close()
print('saved outputs/models/final_02_feature_importance.png')

## 7. Actual vs predicted over the test period (all 100 series summed)

In [ ]:
daily = test[['dt', 'sale_amount']].copy()
daily['Random Forest'] = preds['Random Forest']
daily['Baseline'] = preds['Baseline (prev-week same-day)']
daily = daily.groupby('dt').sum()
plt.figure(figsize=(8, 4))
plt.plot(daily.index, daily['sale_amount'], color='black', lw=2, label='Actual')
plt.plot(daily.index, daily['Random Forest'], color='#2a78d6', lw=2, label='Random Forest')
plt.plot(daily.index, daily['Baseline'], color='#eb6834', lw=2, ls='--', label='Baseline')
plt.ylabel('Total scaled sales'); plt.title('Random Forest tracks actual sales more closely than the baseline')
plt.legend(); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig('outputs/models/final_03_actual_vs_predicted.png', dpi=150); plt.close()
print('saved outputs/models/final_03_actual_vs_predicted.png')

## 8. Phase 5 — Evaluation on the official eval split

The strongest test: the dataset's own **eval split**, the true next 7 days (2024-06-26 to 07-02) for the same 100 series. Models are refit on all 90 days of history, with lag features built across the history->future boundary (no leakage), and scored once.

In [ ]:
eval_raw = pd.read_parquet('../Data/FreshRetailNet-50k/eval.parquet')
eval_raw['dt'] = pd.to_datetime(eval_raw['dt'])
raw_cols = ['store_id', 'product_id', 'dt', 'sale_amount', 'stock_hour6_22_cnt', 'discount',
            'holiday_flag', 'activity_flag', 'precpt', 'avg_temperature', 'avg_humidity', 'avg_wind_level']
our_pairs = set(map(tuple, df[['store_id', 'product_id']].drop_duplicates().values))
eval_raw['pair'] = list(zip(eval_raw['store_id'], eval_raw['product_id']))
eval_sub = eval_raw[eval_raw['pair'].isin(our_pairs)][raw_cols]
base = pd.read_csv('data/subset/freshretail_subset_daily.csv', parse_dates=['dt'])[raw_cols]

full = pd.concat([base, eval_sub], ignore_index=True).sort_values(['store_id', 'product_id', 'dt']).reset_index(drop=True)
gs = full.groupby(['store_id', 'product_id'], observed=True)
full['sales_lag1'] = gs['sale_amount'].shift(1)
full['sales_lag7'] = gs['sale_amount'].shift(7)
full['sales_avg7'] = gs['sale_amount'].transform(lambda s: s.shift(1).rolling(7).mean())
for col in ['stock_hour6_22_cnt', 'avg_temperature', 'avg_humidity', 'avg_wind_level', 'precpt']:
    full['prev_' + col] = gs[col].shift(1)
full['day_of_week'] = full['dt'].dt.day_of_week
full = pd.get_dummies(full, columns=['day_of_week'], prefix='dow', drop_first=True, dtype=int)

cutoff = base['dt'].max()
hist_rows = full[full['dt'] <= cutoff].dropna(subset=features)
eval_rows = full[full['dt'] > cutoff].dropna(subset=features)
lin_f = LinearRegression().fit(hist_rows[features], hist_rows['sale_amount'])
rf_f = RandomForestRegressor(n_estimators=300, max_depth=best_depth, min_samples_leaf=best_leaf,
                             random_state=42, n_jobs=-1).fit(hist_rows[features], hist_rows['sale_amount'])
ye = eval_rows['sale_amount']
ev_rows = []
for name, p in [('Baseline (prev-week)', eval_rows['sales_lag7'].values),
                ('Linear regression', lin_f.predict(eval_rows[features])),
                ('Random Forest', rf_f.predict(eval_rows[features]))]:
    ev_rows.append({'Model': name, **score(ye, p)})
eval_results = pd.DataFrame(ev_rows).set_index('Model').round(3)
print('Official eval week', eval_rows['dt'].min().date(), 'to', eval_rows['dt'].max().date(), '|', len(eval_rows), 'rows')
print(eval_results.to_string())

ax = eval_results[['MAE', 'RMSE']].plot(kind='barh', figsize=(8, 3.5), color=['#bbbbbb', '#2a78d6'])
ax.set_title('Official eval week — error (lower is better)'); ax.invert_yaxis()
plt.tight_layout(); plt.savefig('outputs/models/final_04_eval_split.png', dpi=150); plt.close()
print('saved outputs/models/final_04_eval_split.png')

## 9. Forecasting view (Weeks 3-4)

The models above make **one-day-ahead** predictions. To lean into the *forecasting* framing (estimating future values over a horizon), we add two time-series views on the official eval week:

1. a **recursive multi-step** 7-day forecast (feed each day's prediction back in as the next day's lag), with a prediction-interval band that widens with the horizon;
2. a classical **Holt-Winters** (triple exponential smoothing) weekly forecast on the aggregate daily series, compared with a seasonal-naive baseline.

### (a) Recursive 7-day forecast for an example series

Step forward one day at a time; each prediction becomes tomorrow's `sales_lag1` and feeds the 7-day average. Planned factors (discount/holiday/promotion, day of week) are known in advance; previous-day weather/stockout are taken as the realized prior day (a true single-origin forecast would use a weather forecast). The band is `1.96 * sigma * sqrt(h)`, so uncertainty grows with the horizon `h`.

In [ ]:
resid_sigma = (hist_rows['sale_amount'] - rf_f.predict(hist_rows[features])).std()
dow_cols = [c for c in features if c.startswith('dow_')]
eval_days = sorted(eval_rows['dt'].unique())

def recursive_forecast(sid, pid):
    one = full[(full['store_id'] == sid) & (full['product_id'] == pid)].sort_values('dt').set_index('dt')
    sales = one.loc[one.index <= cutoff, 'sale_amount'].tolist()   # 90 actual days of history
    out = []
    for h, day in enumerate(eval_days, start=1):
        r = one.loc[day]
        x = {c: r[c] for c in dow_cols}
        x.update({'discount': r['discount'], 'activity_flag': r['activity_flag'], 'holiday_flag': r['holiday_flag'],
                  'sales_lag1': sales[-1], 'sales_lag7': sales[-7], 'sales_avg7': np.mean(sales[-7:]),
                  'prev_stock_hour6_22_cnt': r['prev_stock_hour6_22_cnt'], 'prev_avg_temperature': r['prev_avg_temperature'],
                  'prev_avg_humidity': r['prev_avg_humidity'], 'prev_avg_wind_level': r['prev_avg_wind_level'],
                  'prev_precpt': r['prev_precpt']})
        yhat = rf_f.predict(pd.DataFrame([x])[features])[0]
        out.append((day, r['sale_amount'], yhat, 1.96 * resid_sigma * np.sqrt(h)))
        sales.append(yhat)                                         # feed the prediction back in
    return pd.DataFrame(out, columns=['dt', 'actual', 'forecast', 'band']).set_index('dt')

example = eval_rows.groupby(['store_id', 'product_id'])['sale_amount'].mean().idxmax()
fc = recursive_forecast(*example)
print(f'recursive 7-day forecast - store {example[0]}, product {example[1]}:')
print(fc.round(2).to_string())

plt.figure(figsize=(8, 4))
plt.fill_between(fc.index, fc['forecast'] - fc['band'], fc['forecast'] + fc['band'], color='#2a78d6', alpha=0.15, label='95% interval')
plt.plot(fc.index, fc['forecast'], color='#2a78d6', marker='x', label='recursive forecast')
plt.plot(fc.index, fc['actual'], color='black', marker='o', label='actual')
plt.title(f'Recursive 7-day forecast - store {example[0]}, product {example[1]}')
plt.ylabel('sale_amount'); plt.legend(); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig('outputs/models/final_05_recursive_forecast.png', dpi=150); plt.close()
print('saved outputs/models/final_05_recursive_forecast.png')

**Reading it:** the recursive forecast follows the general level but **drifts on sharp swings** - errors compound because each prediction feeds the next - and the 95% band **widens with the horizon**. This is why multi-step forecasts are inherently less certain than the one-day-ahead numbers in sections 5 and 8.

### (b) Holt-Winters (triple exponential smoothing), weekly

Additive **level / trend / seasonal** smoothing (period = 7 days) on the aggregate daily-mean series, implemented from scratch (no `statsmodels`). We grid-search the smoothing weights `alpha, beta, gamma` and forecast the eval week, against a **seasonal-naive** ('repeat last week') baseline.

In [ ]:
agg = df.groupby('dt')['sale_amount'].mean().sort_index()
actual_week = eval_rows.groupby('dt')['sale_amount'].mean().sort_index()

def holt_winters(y, m, alpha, beta, gamma, h):
    level = y[:m].mean()
    trend = (y[m:2*m].mean() - y[:m].mean()) / m
    season = list(y[:m] - level)
    for t in range(len(y)):
        new_level = alpha * (y[t] - season[t % m]) + (1 - alpha) * (level + trend)
        trend = beta * (new_level - level) + (1 - beta) * trend
        season[t % m] = gamma * (y[t] - new_level) + (1 - gamma) * season[t % m]
        level = new_level
    return np.array([level + (k + 1) * trend + season[(len(y) + k) % m] for k in range(h)])

y = agg.values
best = None
for alpha in [0.1, 0.2, 0.3, 0.5]:
    for beta in [0.01, 0.05, 0.1]:
        for gamma in [0.1, 0.3, 0.5]:
            f = holt_winters(y, 7, alpha, beta, gamma, 7)
            err = rmse(actual_week.values, f)
            if best is None or err < best[0]:
                best = (err, alpha, beta, gamma, f)
hw_fc = best[4]; naive = y[-7:]
print(f'Holt-Winters alpha={best[1]}, beta={best[2]}, gamma={best[3]}')
print(f'  Holt-Winters   MAE {mean_absolute_error(actual_week.values, hw_fc):.3f}  RMSE {rmse(actual_week.values, hw_fc):.3f}')
print(f'  seasonal-naive MAE {mean_absolute_error(actual_week.values, naive):.3f}  RMSE {rmse(actual_week.values, naive):.3f}')

plt.figure(figsize=(9, 4))
plt.plot(agg.index[-21:], agg.values[-21:], color='black', label='history (last 3 weeks)')
plt.plot(actual_week.index, actual_week.values, color='black', marker='o', ls='--', label='actual (eval week)')
plt.plot(actual_week.index, hw_fc, color='#2a78d6', marker='x', label='Holt-Winters forecast')
plt.title('Holt-Winters weekly forecast of aggregate daily sales')
plt.ylabel('mean sale_amount'); plt.legend(); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.savefig('outputs/models/final_06_holt_winters.png', dpi=150); plt.close()
print('saved outputs/models/final_06_holt_winters.png')

**Reading it:** Holt-Winters captures the weekly cycle and forecasts the eval week well on the aggregate (MAE ~0.10), but a **seasonal-naive** 'repeat last week' does essentially as well (MAE ~0.09). With only 90 days there is a weekly pattern to exploit but **no annual seasonality**, so classical smoothing adds little over the simple baseline. Note the aggregate of 100 series is far smoother - and easier to forecast - than any single series (part a).

*Takeaway:* this subset supports **short-horizon** forecasting (next-day to ~1 week); a longer, multi-year dataset would be needed for seasonal forecasting.

## 10. Summary

- **Random Forest is the best model**, with linear regression close behind; both beat the previous-week baseline by about 20% on the test period, and the forest wins on the official eval week too.
- **Past sales do most of the work** (recent 7-day average dominates importance); **holidays and discounts** are the most useful factors beyond past sales, while weather adds almost nothing.
- **Long-stockout days are harder** (WAPE improves once the 107 >=12h-stockout test rows are removed), because observed sales there reflect empty shelves, not demand.
- The differences between linear and the tuned forest are small, so a simple, interpretable model is defensible.

**Limitations:** observed (scaled) sales, not unmet demand; only 90 days of history (weekly but not annual seasonality); a 100-series convenience sample that limits generalization.